# Pokemon Analysis

Load the raw data from `data/raw`

## Data and execution notes

Run the cells in order. The existing loading and cleaning steps are retained below.
Dependencies: pandas, openpyxl, matplotlib, seaborn, and Jupyter.

**Data unavailable at the time of this update:** this repository does not include
`data/raw/Pokedex_Cleaned.csv`, `data/raw/PokeStats.csv`, or
`data/raw/Pokedex_Cleaned.xlsx`. Follow `data/README.md` to obtain the dataset.
The loading check reports missing files before the analysis starts. Existing saved
outputs below come from an earlier run; they do not establish current results.
The new analyses and conclusions have not been run on the actual dataset.

In [1]:
import pandas as pd

In [2]:
from pathlib import Path

# Find the project from either its root or the notebooks directory.
# In Colab, upload the files to /content or clone the repository and cd into it.
required_files = ["Pokedex_Cleaned.csv", "PokeStats.csv", "Pokedex_Cleaned.xlsx"]
candidates = [Path.cwd() / "data" / "raw", Path.cwd().parent / "data" / "raw",
              Path("/content")]
RAW_DIR = next((folder for folder in candidates
                if all((folder / name).is_file() for name in required_files)), None)
if RAW_DIR is None:
    details = "\n".join(
        f"{folder}: missing {', '.join(name for name in required_files if not (folder / name).is_file())}"
        for folder in candidates
    )
    raise FileNotFoundError(
        "Required project data were not found. Download the dataset as described "
        "in data/README.md and place the files in data/raw (or /content in Colab).\n"
        + details
    )
print(f"Loading data from: {RAW_DIR}")

In [3]:
pokedex = pd.read_csv(f"{RAW_DIR}/Pokedex_Cleaned.csv", encoding="latin1")
pokedex.head()

,#,Name,Primary Type,Secondary Type,Total,HP,Attack,Defense,Sp.Atk,Sp.Def,Speed,Variant
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,NaN
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,NaN
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,NaN
3,3,Venusaur,Grass,Poison,625,80,100,123,122,120,80,NaN
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,NaN


In [4]:
pokedex_xlsx = pd.read_excel(f"{RAW_DIR}/Pokedex_Cleaned.xlsx")
pokedex_xlsx.head()

,#,Name,Primary Type,Secondary Type,Total,HP,Attack,Defense,Sp.Atk,Sp.Def,Speed,Variant
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,NaN
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,NaN
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,NaN
3,3,Venusaur,Grass,Poison,625,80,100,123,122,120,80,NaN
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,NaN


In [5]:
pokestats = pd.read_csv(f"{RAW_DIR}/PokeStats.csv")
pokestats.head()

,b'# Name Type Total HP Attack Defense Sp. Atk Sp. Def Speed'
0,b'001 BulbasaurGrass Poison\n318\n45\n49\n49\n...
1,b'002 IvysaurGrass Poison\n405\n60\n62\n63\n80...
2,b'003 VenusaurGrass Poison\n525\n80\n82\n83\n1...
3,b'003 Venusaur Mega VenusaurGrass Poison\n625\...
4,b'004 CharmanderFire \n309\n39\n52\n43\n60\n50...


## Data Cleaning

In [6]:
pokedex.info()

<class 'pandas.DataFrame'>
RangeIndex: 1173 entries, 0 to 1172
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   #               1173 non-null   int64
 1   Name            1173 non-null   str  
 2   Primary Type    1173 non-null   str  
 3   Secondary Type  633 non-null    str  
 4   Total           1173 non-null   int64
 5   HP              1173 non-null   int64
 6   Attack          1173 non-null   int64
 7   Defense         1173 non-null   int64
 8   Sp.Atk          1173 non-null   int64
 9   Sp.Def          1173 non-null   int64
 10  Speed           1173 non-null   int64
 11  Variant         190 non-null    str  
dtypes: int64(8), str(4)
memory usage: 110.1 KB


In [7]:
pokedex.shape

(1173, 12)

In [8]:
pokedex.isnull().sum()


#                   0
Name                0
Primary Type        0
Secondary Type    540
Total               0
HP                  0
Attack              0
Defense             0
Sp.Atk              0
Sp.Def              0
Speed               0
Variant           983
dtype: int64

In [9]:
pokedex.duplicated().sum()

np.int64(1)

In [10]:
pokestats.info()

<class 'pandas.DataFrame'>
RangeIndex: 1190 entries, 0 to 1189
Data columns (total 1 columns):
 #   Column                                                        Non-Null Count  Dtype
---  ------                                                        --------------  -----
 0   b'# Name Type Total HP Attack Defense Sp. Atk Sp. Def Speed'  1190 non-null   str  
dtypes: str(1)
memory usage: 9.4 KB


In [11]:
pokestats.shape

(1190, 1)

In [12]:
pokestats.isnull().sum()

b'# Name Type Total HP Attack Defense Sp. Atk Sp. Def Speed'    0
dtype: int64

In [13]:
pokestats.duplicated().sum()

np.int64(0)

In [14]:
# Found that pokestats is being read as 1 column

with open(f"{RAW_DIR}/PokeStats.csv", "r", encoding="latin1") as file:
    for i in range(5):
        print(repr(file.readline()))

"b'# Name Type Total HP Attack Defense Sp. Atk Sp. Def Speed'\n"
"b'001 BulbasaurGrass Poison\\n318\\n45\\n49\\n49\\n65\\n65\\n45'\n"
"b'002 IvysaurGrass Poison\\n405\\n60\\n62\\n63\\n80\\n80\\n60'\n"
"b'003 VenusaurGrass Poison\\n525\\n80\\n82\\n83\\n100\\n100\\n80'\n"
"b'003 Venusaur Mega VenusaurGrass Poison\\n625\\n80\\n100\\n123\\n122\\n120\\n80'\n"


In [15]:
import ast
import pandas as pd

type_names = {
    "Normal", "Fire", "Water", "Electric", "Grass", "Ice",
    "Fighting", "Poison", "Ground", "Flying", "Psychic",
    "Bug", "Rock", "Ghost", "Dragon", "Dark", "Steel", "Fairy"
}

rows = []

with open(f"{RAW_DIR}/PokeStats.csv", "r", encoding="latin1") as file:
    for raw_line in file:
        if not raw_line.strip():
            continue

        decoded = ast.literal_eval(
            raw_line.strip()
        ).decode("utf-8", errors="replace")

        parts = decoded.splitlines()

        if parts[0].strip().startswith("#"):
            continue

        first_line = parts[0].strip()
        number = first_line[:3]
        remaining = first_line[3:].strip()
        types = []

        for _ in range(2):
            found_type = None

            for pokemon_type in sorted(
                type_names,
                key=len,
                reverse=True
            ):
                if remaining.endswith(pokemon_type):
                    found_type = pokemon_type
                    remaining = remaining[
                        :-len(pokemon_type)
                    ].strip()
                    break

            if found_type is None:
                break

            types.insert(0, found_type)
        name = remaining
        stats = [
            int(value.strip())
            for value in parts[1:]
            if value.strip()
        ]

        if len(stats) >= 7:
            rows.append([
                number,
                name,
                types[0] if len(types) > 0 else None,
                types[1] if len(types) > 1 else None,
                *stats[:7]
            ])

pokestats = pd.DataFrame(rows, columns=[
    "#", "Name", "Type 1", "Type 2",
    "Total", "HP", "Attack", "Defense",
    "Sp. Atk", "Sp. Def", "Speed"
])

pokestats.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed
0,001,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45
1,002,Ivysaur,Grass,Poison,405,60,62,63,80,80,60
2,003,Venusaur,Grass,Poison,525,80,82,83,100,100,80
3,003,Venusaur Mega Venusaur,Grass,Poison,625,80,100,123,122,120,80
4,004,Charmander,Fire,NaN,309,39,52,43,60,50,65


In [16]:
pokestats.info()

<class 'pandas.DataFrame'>
RangeIndex: 1190 entries, 0 to 1189
Data columns (total 11 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   #        1190 non-null   str  
 1   Name     1190 non-null   str  
 2   Type 1   1190 non-null   str  
 3   Type 2   648 non-null    str  
 4   Total    1190 non-null   int64
 5   HP       1190 non-null   int64
 6   Attack   1190 non-null   int64
 7   Defense  1190 non-null   int64
 8   Sp. Atk  1190 non-null   int64
 9   Sp. Def  1190 non-null   int64
 10  Speed    1190 non-null   int64
dtypes: int64(7), str(4)
memory usage: 102.4 KB


In [17]:
# checking to see if it worked
pokestats.shape

(1190, 11)

In [18]:
# removed duplicate pokemon records
pokedex = pokedex.drop_duplicates().copy()
pokestats = pokestats.drop_duplicates().copy()

#replaced missing "Secondary Types" to None
pokedex["Secondary Type"] = pokedex["Secondary Type"].fillna("None")
#replaced missing "Variants" to None
pokedex["Variant"] = pokedex["Variant"].fillna("None")
#replaced missing "Type 2" to None
pokestats["Type 2"] = pokestats["Type 2"].fillna("None")

stat_columns = [
    "Total", "HP", "Attack", "Defense",
    "Sp. Atk", "Sp. Def", "Speed"
]

for column in stat_columns:
    pokestats[column] = pd.to_numeric(
        pokestats[column], errors="coerce"
    )

In [19]:
# Optional: save cleaned files locally. Analysis does not require these exports.
SAVE_CLEANED_CSV = False
if SAVE_CLEANED_CSV:
    pokedex.to_csv(f"{RAW_DIR}/Pokedex_Cleaned_Final.csv", index=False)
    pokestats.to_csv(f"{RAW_DIR}/PokeStats_Cleaned_Final.csv", index=False)

## Exploratory Data Analysis

Use the cleaned `pokedex` DataFrame for all three questions. Convert its stat
columns to numbers, report missing values, and retain alternate forms as separate
entries. Means and sample standard deviations skip missing values; the tables
include counts so the amount of usable data is visible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")
analysis_stats = ["HP", "Attack", "Defense", "Sp.Atk", "Sp.Def", "Speed"]
required_columns = ["Primary Type", "Total"] + analysis_stats
missing_columns = [column for column in required_columns if column not in pokedex.columns]
if missing_columns:
    raise ValueError(f"Missing columns in pokedex: {missing_columns}")

# The original cleaning converts pokestats; these conversions prepare pokedex.
for column in ["Total"] + analysis_stats:
    pokedex[column] = pd.to_numeric(pokedex[column], errors="coerce")

# Blank type labels should not become their own group.
pokedex["Primary Type"] = pokedex["Primary Type"].str.strip().replace("", pd.NA)
display(pokedex.describe())
display(pokedex["Primary Type"].value_counts(dropna=False).rename("Entry count"))
display(pokedex["Total"].agg(["min", "max", "mean"]).rename("Total stats"))
display(pokedex.isna().sum().rename("Missing values after cleaning"))
print("Duplicate rows after cleaning:", pokedex.duplicated().sum())
print("The cleaning step represents missing secondary types and variants as 'None'.")

## Research Question 1: Average Total Stats by Primary Type

Group entries by primary type and compare their average Total stats. These are
averages per dataset entry, so species with multiple forms can contribute more
than once.

In [ ]:
type_means = pokedex.groupby("Primary Type")["Total"].mean().dropna().sort_values(ascending=False)
display(pd.DataFrame({"Mean Total": type_means,
                      "Valid Total count": pokedex.groupby("Primary Type")["Total"].count()}))
if type_means.empty:
    raise ValueError("No valid type and Total pairs are available for Question 1.")

fig, ax = plt.subplots(figsize=(11, 5))
type_means.plot.bar(ax=ax, color=sns.color_palette("viridis", len(type_means)))
ax.set(title="Average Total Stats by Primary Type", xlabel="Primary type", ylabel="Mean Total stats")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# List all tied groups, if there is a tie.
highest_types = ", ".join(type_means[type_means == type_means.max()].index)
lowest_types = ", ".join(type_means[type_means == type_means.min()].index)
print(f"Highest: {highest_types} ({type_means.max():.2f})")
print(f"Lowest: {lowest_types} ({type_means.min():.2f})")

## Research Question 2: Which Stats Vary the Most?

Sample standard deviation (`std()`, with `ddof=1`) measures how spread out a stat
is around its mean. All six stats use stat points, so their standard deviations
can be compared directly. At least two nonmissing values are needed per stat.

In [ ]:
stat_std = pokedex[analysis_stats].std().sort_values(ascending=False)
display(pd.DataFrame({"Standard deviation": stat_std,
                      "Valid count": pokedex[analysis_stats].count()}))
if stat_std.isna().any():
    raise ValueError("Each of the six stats needs at least two valid values for Question 2.")

fig, ax = plt.subplots(figsize=(9, 5))
stat_std.plot.bar(ax=ax, color=sns.color_palette("mako", len(stat_std)))
ax.set(title="Variation in Pokémon Stats", xlabel="Statistic", ylabel="Sample standard deviation (stat points)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

most_variable = ", ".join(stat_std[stat_std == stat_std.max()].index)
least_variable = ", ".join(stat_std[stat_std == stat_std.min()].index)
print(f"Most variation: {most_variable} ({stat_std.max():.2f})")
print(f"Least variation: {least_variable} ({stat_std.min():.2f})")

## Research Question 3: Average Total Stats Across Generations 1–9

Use `Generation` when present. Otherwise, use the National Pokédex number (`#`)
and these inclusive ranges: 1–151, 152–251, 252–386, 387–493, 494–649,
650–721, 722–809, 810–905, and 906–1025.
[Range reference: Serebii's National Pokédex generation navigation](https://www.serebii.net/pokemon/gen9pokemon.shtml).
Numbers outside these ranges remain unassigned and are reported. A form is
assigned to its species' generation by number, which may differ from the
generation in which that form was introduced. A represented generation does
not necessarily mean every species from that generation is included.

In [ ]:
if "Generation" in pokedex.columns:
    generation = pd.to_numeric(pokedex["Generation"], errors="coerce")
    generation = generation.where(generation.isin(range(1, 10)))
else:
    if "#" not in pokedex.columns:
        raise ValueError("Question 3 requires a Generation column or National Pokédex '#' column.")
    national_number = pd.to_numeric(pokedex["#"], errors="coerce")
    national_number = national_number.where(national_number.mod(1).eq(0))
    generation = pd.cut(national_number,
                        bins=[0, 151, 251, 386, 493, 649, 721, 809, 905, 1025],
                        labels=range(1, 10))

# Use a separate table so the source generation labels remain unchanged.
generation_data = pd.DataFrame({"Generation": generation, "Total": pokedex["Total"]})
generation_counts = generation_data.groupby("Generation", observed=True).size().reindex(range(1, 10), fill_value=0)
generation_means = generation_data.groupby("Generation", observed=True)["Total"].mean().reindex(range(1, 10))
valid_totals = generation_data.groupby("Generation", observed=True)["Total"].count().reindex(range(1, 10), fill_value=0)
display(pd.DataFrame({"Entry count": generation_counts, "Valid Total count": valid_totals,
                      "Mean Total": generation_means}))
missing_generations = generation_counts[generation_counts == 0].index.tolist()
print("Generations without entries:", missing_generations or "None; all nine are represented.")
print("Entries with unassigned generation:", generation.isna().sum())
if generation_means.notna().sum() == 0:
    raise ValueError("No valid generation and Total pairs are available for Question 3.")

fig, ax = plt.subplots(figsize=(10, 5))
generation_means.plot.bar(ax=ax, color=sns.color_palette("crest", 9))
for position, value in enumerate(generation_means):
    if pd.isna(value):
        ax.text(position, 0, "No data", rotation=90, ha="center", va="bottom")
ax.set(title="Average Total Stats by Generation", xlabel="Generation", ylabel="Mean Total stats")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

highest_generations = ", ".join(str(g) for g in generation_means[generation_means == generation_means.max()].index)
lowest_generations = ", ".join(str(g) for g in generation_means[generation_means == generation_means.min()].index)
print(f"Highest: generation {highest_generations} ({generation_means.max():.2f})")
print(f"Lowest: generation {lowest_generations} ({generation_means.min():.2f})")

## Conclusions

**Pending actual data:** the required files are absent from this checkout.
No numerical findings are claimed here. Run all cells after adding the files;
the following cell displays conclusions calculated from those data.

- **Question 1 — placeholder:** highest and lowest average primary types and their means are pending.
- **Question 2 — placeholder:** most and least variable statistics and their standard deviations are pending.
- **Question 3 — placeholder:** highest and lowest generation means, and whether all nine generations are represented, are pending.

Alternate Pokémon forms may appear as separate dataset entries and can affect
all three comparisons. Exact duplicate removal does not combine distinct forms.
These descriptive results summarize the available entries and do not explain
why a type or generation has higher stats.

In [ ]:
coverage = ("All nine generations have entries in this dataset." if not missing_generations
            else f"Generations {', '.join(map(str, missing_generations))} have no entries in this dataset.")
display(Markdown(f"""
### Calculated conclusions (current run)

1. **Primary types:** {highest_types} has the highest mean Total ({type_means.max():.2f}),
   while {lowest_types} has the lowest ({type_means.min():.2f}). These averages compare
   the overall stat totals of the available entries in each primary type.
2. **Stat variation:** {most_variable} varies most (sample standard deviation
   {stat_std.max():.2f} stat points), while {least_variable} varies least
   ({stat_std.min():.2f} stat points). A larger standard deviation means entries
   differ more in that statistic.
3. **Generations:** Generation {highest_generations} has the highest mean Total
   ({generation_means.max():.2f}), and generation {lowest_generations} has the lowest
   ({generation_means.min():.2f}) among generations with usable data. {coverage}
   There are {generation.isna().sum()} entries with unassigned generations.
   Missing averages are not zero, and these comparisons do not establish a trend over time.

Distinct alternate forms are counted separately and can change these averages
and standard deviations. Missing numeric values are excluded from each calculation.
"""))